# Competition Notebook

This notebook will implement the competition requirements steps by steps

From the project description, below is the background of the task to be understood and implemented here.

## Background (fictional)

Marvin works at the last Blockbusta Videoz (a fictional video rental shop) where his task is to classify movie and TV show reviews to help curate a special section of the store. One day while perusing his favorite website, Marvin came upon a post about someone who secretly automated their job and then quietly took off on a long (paid) vacation. Inspired, Marvin set a side a portion of his salary to hire a developer to write a few scripts to scrape movie reviews from various sources. It's a bit noisy, but aggregating many reviews has cut the time he previously spent on his job by half. Now, after reading an article about AI, Marvin wants to take things a step further: he is searching for a program that can determine a) whether or not a piece of text is a movie/TV show review and b) whether or not each review is positive (the movie/TV show is recommended) or negative (the movie/TV show should be avoided). Marvin has put together a competition and advertised it on Fraggle (a fictional platform for competitive data science).

### End-goal for this project
The task here is to classify the documents as one of the following categories:

1. Not a (movie/TV show) review
2. Positive (movie/TV show) review
3. Negative (movie/TV show) review

In [1]:
NAME = "Mohammad Rahat Helal"
# University of Arizona email address
EMAIL = "mohammadrhelal@arizona.edu"
# Names of any collaborators.  Write N/A if none.
COLLABORATORS = "N/A"

### Note: 
It was observed that the current docker image used for this competition does not have any Python or scikit-learn libraries installed, so installing them here before using them in subsequent steps

In [2]:
# need to run this piece before importing them in next steps


!pip install pandas scikit-learn

     |████████████████████████████████| 11.3 MB 2.0 MB/s eta 0:00:01    |▏                               | 51 kB 1.9 MB/s eta 0:00:06     |██████████████████████████████▋ | 10.8 MB 2.0 MB/s eta 0:00:01
     |████████████████████████████████| 24.8 MB 10.9 MB/s eta 0:00:01.5 MB 10.9 MB/s eta 0:00:0300:02     |█████████████████▊              | 13.8 MB 10.9 MB/s eta 0:00:02
     |████████████████████████████████| 38.1 MB 3.5 MB/s eta 0:00:011    |█████▊                          | 6.8 MB 18.7 MB/s eta 0:00:02██▌                         | 7.8 MB 18.7 MB/s eta 0:00:02  | 19.7 MB 18.7 MB/s eta 0:00:01�███████████████████▋          | 25.7 MB 18.7 MB/s eta 0:00:01███████████████████▌  | 35.1 MB 3.5 MB/s eta 0:00:01
     |████████████████████████████████| 302 kB 33.9 MB/s eta 0:00:01     |█████████████████████▊          | 204 kB 33.9 MB/s eta 0:00:01


In [3]:
import pandas as pd

### Read data 

The three csv files resides in the data folder and can be read into a pandas dataframe to be used in subsequent steps

In [4]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")

# check the shape to confirm they are as described in the problem statement
print(train.shape)
print(test.shape)
print(sample.shape)

train.head()

(70317, 3)
(17580, 2)
(17580, 2)


,ID,TEXT,LABEL
0,327995652116863138,Carolyn Baker defines the niche of helping peo...,0
1,11848336500074516230,Just getting released from a six month drug re...,1
2,8453485550425672763,I was greatly dissappointed when I popped this...,0
3,13088897910749354342,This is a film that has garnered any interest ...,2
4,4199320520317837800,This is one of the more adorable episodes of t...,1


In [5]:
# check the field names to ensure they are as described

print(train.columns)
print(test.columns)
print(sample.columns)

Index(['ID', 'TEXT', 'LABEL'], dtype='object')
Index(['ID', 'TEXT'], dtype='object')
Index(['ID', 'LABEL'], dtype='object')


In [6]:
# let us check the label disctribution to see if this is unbalanced

train["LABEL"].value_counts()

0    32289
1    19139
2    18889
Name: LABEL, dtype: int64

In [7]:
test.head()

,ID,TEXT
0,1087873697464833975,This tv series is one of the best I have seen ...
1,5853461517618045821,I saw Evanescence live this summer and it was ...
2,1225516091890726770,Almost from the word go this film is poor and ...
3,11795874926011119457,I bought this book and I know friends who have...
4,15956464546963841646,"“As you please.”\n\n“Ah, I forgot! A letter ca..."


In [8]:
sample.head()

,ID,LABEL
0,1087873697464833975,1
1,5853461517618045821,1
2,1225516091890726770,1
3,11795874926011119457,1
4,15956464546963841646,2


### Split source data

First Step is to create a training and validation split of source data

In [9]:
from sklearn.model_selection import train_test_split

X = train["TEXT"].fillna("")
y = train["LABEL"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Create a base model

Next we create a base model using sklearn library

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),   # unigrams + bigrams
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=3.0,
        solver="liblinear"
    ))
])

model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2),
                                 stop_words='english', sublinear_tf=True)),
                ('clf',
                 LogisticRegression(C=3.0, class_weight='balanced',
                                    max_iter=3000, solver='liblinear'))])

### Model evaluation

Next the model is evaluated for accuracy and F1 scores

In [11]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

valid_preds = model.predict(X_valid)

print("Accuracy:", accuracy_score(y_valid, valid_preds))
print("Macro F1:", f1_score(y_valid, valid_preds, average="macro"))

print(classification_report(y_valid, valid_preds))

Accuracy: 0.9281143344709898
Macro F1: 0.9163585974341654
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      6458
           1       0.88      0.88      0.88      3828
           2       0.91      0.87      0.89      3778

    accuracy                           0.93     14064
   macro avg       0.92      0.91      0.92     14064
weighted avg       0.93      0.93      0.93     14064



### Model train on full dataset

Here we use the full data to train our final model before we predict test data

In [12]:
final_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=3.0,
        solver="liblinear"
    ))
])

final_model.fit(train["TEXT"].fillna(""), train["LABEL"])

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2),
                                 stop_words='english', sublinear_tf=True)),
                ('clf',
                 LogisticRegression(C=3.0, class_weight='balanced',
                                    max_iter=3000, solver='liblinear'))])

### Prediction on Test data

In [13]:
test_preds = final_model.predict(test["TEXT"].fillna(""))

### Create the submission csv file

In [14]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "LABEL": test_preds
})

submission.to_csv("submission.csv", index=False)

In [15]:
submission.head()
submission["LABEL"].value_counts()

0    8072
1    4932
2    4576
Name: LABEL, dtype: int64

In [26]:
submission.head()

,ID,LABEL
0,1087873697464833975,1
1,5853461517618045821,1
2,1225516091890726770,2
3,11795874926011119457,0
4,15956464546963841646,0


In [27]:
print(submission.shape)

(17580, 2)


In [23]:
print(submission.columns)

Index(['ID', 'LABEL'], dtype='object')


In [19]:
submission["LABEL"].unique()

array([1, 2, 0])

In [20]:
(submission["ID"] == test["ID"]).all()

True

In [21]:
print(submission.shape)

(17580, 2)


In [22]:
submission.to_csv("submission.csv", index=False)

In [24]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "LABEL": test_preds
})

submission.to_csv("submission.csv", index=False)

In [25]:
submission["ID"].duplicated().sum()

0